# Evaluation

<a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> ended on a question. Swapping the embedder moved the answer to *How do you win
a game of checkers?* up or down the rankings, but should you switch models on the strength of
two questions? This notebook answers it the standard way: write down the right answers for a
set of questions once, then score any retrieval setup against them.

1. **Golds**: questions with their right answers, written down once.
2. **hit@k and MRR**: two scores that compare embedders with numbers.
3. **Poker**: a harder rule set, where retrieval's mistakes are harder to spot.
4. **A hosted embedder**, and the app's other kind of gold.

You don't need to have run the other notebooks first. This notebook runs **key-free**: everything uses small local models, and nothing here calls
generation, so nothing is billed.

On Colab, run the cells from the top; the first one fetches the repo (about a minute). On your
own machine, run `uv run jupyter lab` from the repo root.

In [ ]:
# On Colab, fetch the repo first (about a minute). On your own machine this is skipped.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/ecoop/rulebook /content/rulebook
    %pip install -qq -e /content/rulebook
    sys.path.insert(0, "/content/rulebook/src")  # this session can't see the new install otherwise

from rulebook.notebook import hit_cell, setup, show_table
setup()

## 1. Three embedders to compare

The same three local models as in <a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> (§5): small *static* **model2vec "potion"**
models from [MinishLab](https://huggingface.co/minishlab), 2M, 8M and 32M parameters, the
largest tuned for retrieval. Each gets its own temporary index of the checkers rules, so your
real index isn't touched. (The first run downloads the models, a few tens of MB.)

In [ ]:
from rulebook.config import settings
from rulebook.eval import build_scratch_index

MODELS = [
    "minishlab/potion-retrieval-32M",
    "minishlab/potion-base-8M",
    "minishlab/potion-base-2M",
]
indexes = {m: build_scratch_index("checkers", m) for m in MODELS}


def label(model):
    name = model.split("/")[-1]
    return f"{name} (default)" if model == settings.embedding_model else name

## 2. Write the answers down once: golds

When you look at a retrieval result and see that it's right, you're checking it against an
answer you already know. A **gold** is that knowledge written down: a question paired with its right
answer. Here the answer is given as **citations** — the places in the rules that answer it,
written the way the app cites them (`checkers 5.3`) — so these are **citation golds**. Write a
set of them once, and you can score *any* retrieval setup against it — automatically, and the
same way every time.

Here's a minimal gold set for checkers. Each line of `eval/gold/checkers.jsonl` is one gold:

```json
{"domain": "checkers", "question": "How do you win a game of checkers?", "citations": ["checkers 5.1", "checkers 2.1"], "style": "literal"}
```

In [ ]:
from rulebook.eval import load_gold

gold = load_gold("checkers")
show_table(["question", "citations"], [(g.question, " / ".join(g.citations)) for g in gold])

Some golds cite two rules. *How do you win?* is answered by both **2.1 Object of the game** and
**5.1 Winning**, which say the same thing in different words. Retrieving either one counts as
finding the answer, so a model isn't marked wrong for picking the other right rule.

## 3. Score them: hit@k and MRR

Two standard scores turn "did retrieval find it?" into numbers:

- **hit@k** — the share of questions with a cited rule in the top *k*. *Is the answer
  even in what the LLM gets to read?*
- **MRR** (mean reciprocal rank) — score each question 1 / (rank of the first cited rule): a
  rule at rank 1 scores 1.0, rank 2 scores 0.5, rank 3 scores 0.33, and a miss scores 0. Then
  average. It rewards putting the answer *first*, not just somewhere.

In the tables below, each hit@k cell also shows the count behind it: **0.50 (6/12)** means 6 of
the 12 questions had a cited rule in the top *k*. MRR has no count; it's an average of the
per-question scores.

The same `evaluate()` runs here and behind `python -m rulebook.eval`, so the notebook and the app
report the same numbers.

In [ ]:
from rulebook.eval import evaluate

rows = []
for m in MODELS:
    res = evaluate(gold, k=5, index=indexes[m])
    rows.append([label(m)] + [hit_cell(res, k) for k in (1, 3, 5)] + [f"{res.mrr:.2f}"])
show_table(["model", "hit@1", "hit@3", "hit@5", "MRR"], rows)

What the two questions in <a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> couldn't tell you:

- **"Better" isn't one number.** The smallest model puts a cited rule *first* more often than
  the default (higher hit@1), yet finds one in the top 5 no more often (same hit@5). Which
  column matters depends on `k`: this app hands the LLM 5 passages, so a rule at #4 is fine and
  a rule that's missing entirely is not.
- **Bigger, or "tuned for retrieval," isn't automatically better.** Check where the
  retrieval-tuned model lands before assuming it wins.
- **Mind the sample size.** With 12 questions, one question moves hit@k by about 0.08 (1/12). A gold set
  doesn't have to be big to be useful, but it's evidence, not proof.

## 4. A harder rule set: poker

Each checkers rule sounds different from the others. The repo's third sample,
`rules/poker/poker.md`, is built the other way. It covers five variants (five-card draw,
five-card stud, seven-card stud, Razz and Omaha) whose rules use nearly the same words: the
deal, the bring-in, betting rounds, the showdown. They differ in the details. Razz is played
for the *lowest* hand, so its bring-in is the reverse of seven-card stud's. Omaha makes you use
exactly two of your four hole cards. Every rule names its variant ("In Razz, ..."), but
retrieval still has to pick the right one out of five near-copies.

Poker has its own gold set, `eval/gold/poker.jsonl` (22 questions). Score it with the same three
models, next to checkers:

In [ ]:
poker_gold = load_gold("poker")
poker_indexes = {m: build_scratch_index("poker", m) for m in MODELS}

rows = []
for m in MODELS:
    for rule_set, golds, index in (("checkers", gold, indexes[m]), ("poker", poker_gold, poker_indexes[m])):
        res = evaluate(golds, k=5, index=index)
        rows.append([label(m), rule_set] + [hit_cell(res, k) for k in (1, 3, 5)] + [f"{res.mrr:.2f}"])
show_table(["model", "rule set", "hit@1", "hit@3", "hit@5", "MRR"], rows)

By these numbers poker is no harder than checkers. To see what's different, look at the misses
themselves: the questions where the default model's first passage isn't a cited rule, and
which variant that passage comes from.

In [ ]:
VARIANT = {"1": "five-card draw", "2": "five-card stud", "3": "seven-card stud", "4": "Razz", "5": "Omaha"}


def named(rule_id):
    """'4.4' -> '4.4 (Razz)'."""
    variant = VARIANT.get(rule_id.split(".")[0])
    return f"{rule_id} ({variant})" if variant else rule_id


res = evaluate(poker_gold, k=5, index=poker_indexes[MODELS[0]])  # MODELS[0] is the app's default
rows = []
for r in res.items:
    first = r.retrieved_ids[0]
    if first in r.item.rule_ids:
        continue  # retrieval got this one right
    rank = next((n for n, rid in enumerate(r.retrieved_ids, start=1) if rid in r.item.rule_ids), None)
    rows.append([
        r.item.question,
        " / ".join(named(rid) for rid in r.item.rule_ids),
        named(first),
        f"#{rank}" if rank else "not in top 5",
    ])
show_table(["question", "cited", "came first", "cited rule's rank"], rows)

Apart from one preamble (the file's introduction, which also tripped up checkers in
<a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> §4), two kinds of miss show up:

- **Another rule from the right variant.** Most rows are like this: asked about Omaha, retrieval
  returns an Omaha rule, just not the one that answers. Usually the cited rule comes second, so
  with 5 passages Claude still gets to read it; once it isn't in the top 5 at all.
- **A rule from the wrong variant.** *How many betting rounds are there in seven-card
  stud?* gets **4.4 (Razz)** first. Razz's rule says it has five rounds "exactly as in
  seven-card stud", so it contains the question's words. Here the borrowed rule happens to give
  the right number. On the bring-in it wouldn't: in seven-card stud the *lowest* up-card brings
  in, and in Razz the *highest*. *How is the deal structured in five-card stud?* is worse off: a
  seven-card stud rule comes first, and the cited rule isn't in the top 5.

That checkers preamble was plainly off-topic: an LLM reading it can tell it
doesn't answer the question. A rule from the wrong variant is on-topic and reads like an answer,
so nothing downstream flags it. The gold set catches it, but only if you look at which rule came
first and not just the score. One fix is a later stage: a **reranker** reads the question
together with each passage, rather than comparing vectors. That is the next notebook, <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a>.

## 5. Local vs. a hosted embedder (Voyage)

The same gold set answers *"is a hosted model worth it?"* with a number. Hosted models aren't
local, so this one uses the real index (needs a `VOYAGE_API_KEY`, which costs a little):

```bash
# 1. rebuild the checkers index with Voyage vectors
EMBEDDING_PROVIDER=voyage EMBEDDING_MODEL=voyage-4 \
    uv run python scripts/build_index.py --domain checkers
# 2. score it against the same gold set
EMBEDDING_PROVIDER=voyage EMBEDDING_MODEL=voyage-4 \
    uv run python -m rulebook.eval --domain checkers
# 3. put the default back when you're done
uv run python scripts/build_index.py --domain checkers
```

Compare its scores with the table above. On a small, clean corpus like this, a hosted model may
not win by much — and that's a useful result too. Either way it's evidence instead of a guess.

## 6. The other kind of gold: written golds

Everything above used **citation golds**: the answer points into the rules. The full app (the
web UI) also has **written golds**: answers written out as text by a signed-in user, usually to
fix or extend an answer the app gave. They can cover what the rules don't — *"I have 10 players:
ultimate or goaltimate?"* or *"What does it mean to play 'savage'?"*

| | citation gold | written gold |
|---|---|---|
| The answer is | places in the rules (`checkers 5.1`) | text |
| Written by | whoever builds the eval | users, in the app |
| Stored in | `eval/gold/<domain>.jsonl` | the app's gold log, curated by an admin |
| Used for | scoring retrieval (this notebook) | answering: indexed next to the rules, so similar questions retrieve it |

They're the same idea — a question with its right answer — and
[#254](https://github.com/ecoop/rulebook/issues/254) brings them into one shape, so a written
gold can carry citations too. One thing to keep in mind when you get there: a written gold's
own text is always the nearest match to its own question, so retrieval eval scores against the
rules, not against written golds.

## Takeaway

- A **gold** is a question with its right answer written down once. With **citation golds**,
  **hit@k** and **MRR** score any setup against them, so "is this change better?" gets a number.
- Measure before you upgrade: the bigger or fancier model isn't always the better one.
- Read the misses, not only the scores. On poker, a miss can be the same rule from the wrong
  variant, which reads like a real answer.
- The app's **written golds** are the same idea with a text answer; they help answer questions
  rather than score retrieval.

**Next: <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a>** (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/reranking.ipynb" target="_parent">run it in Colab</a>). Why retrieval can't tell the right
variant from the wrong one, and a second stage that reads the question and each passage together.